# back-fn-call-with-recipe-args — ex3: validate grad_out.shape == node.array.shape before invoking back_fn

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `back-fn-call-with-recipe-args`. Running the final beacon cell reports progress against the `Backprop: back fn call with recipe args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: back fn call with recipe args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`back-fn-call-with-recipe-args`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "back-fn-call-with-recipe-args"
DD_SUBTOPIC = "Backprop: back fn call with recipe args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Pre-call shape validation — fail loudly before the back_fn runs

Ex1 invoked the back_fn with the canonical four-arg shape; ex2 recorded
what each channel received. The deepening move is to ASSERT shape
compatibility BEFORE the call. If `grad_out.shape != node.array.shape`,
we know the dispatcher already broke the chain rule — raise a precise
error naming both shapes instead of letting the back_fn produce a
cryptic broadcast/elementwise failure deep inside its math.

```python
def call_back_fn_validated(back_fn, grad_out, node):
    if grad_out.shape != node.array.shape:
        raise ValueError(
            f'grad_out.shape {tuple(grad_out.shape)} != node.array.shape '
            f'{tuple(node.array.shape)} — upstream dispatcher bug'
        )
    return back_fn(grad_out, node.array,
                   *node.recipe.args, **node.recipe.kwargs)
```

**Why it's worth a guard.** In a real autograd, a `grad_out`/`out`
shape mismatch typically surfaces as `RuntimeError: The size of tensor
a (5) must match the size of tensor b (3) at non-singleton dimension
1` inside someone else's elementwise op — six call-frames deep, no
context about which node failed. A pre-call assert pinpoints the
dispatcher as the culprit immediately.

**`tuple(shape)` for the message.** `torch.Size([3, 5])` reprs as
`torch.Size([3, 5])` which is verbose. Tupling first gives `(3, 5)`.

### Exercise 3 — validate grad_out.shape == node.array.shape before invoking back_fn

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a pre-call shape guard that raises a `ValueError` naming both `grad_out.shape` and `node.array.shape` before invoking the back_fn — pinpointing the dispatcher rather than letting an elementwise broadcast error surface from deep inside the back_fn.
> Keywords: validation, shape-check, guard, back-fn-call
> ```

**KCs targeted:** `back-fn-call-with-recipe-args`, `precall-shape-guard`

Implement `ex3_call_back_fn_validated(back_fn, grad_out, node)`. Same canonical four-arg invocation as ex1, plus a pre-call shape guard:

1. If `grad_out.shape != node.array.shape`, raise `ValueError('grad_out.shape <A> != node.array.shape <B> — upstream dispatcher bug')` where `<A>` is `tuple(grad_out.shape)` and `<B>` is `tuple(node.array.shape)`.
2. Otherwise call `back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)` and return its result.

Shapes are the ONLY check — DON'T also validate `recipe.args` or `recipe.kwargs` (out of scope).

Why `tuple(shape)` and not raw `torch.Size`. `torch.Size([3, 5])` reprs as `torch.Size([3, 5])` — verbose. The test asserts both tuples appear in the message text.

Node has the standard MiniTensor shape:
- `node.array` — a `torch.Tensor` (the cached forward output).
- `node.recipe.args` — a tuple of raw tensors.
- `node.recipe.kwargs` — a dict.

The test provides a `Node` namespace class for you.

In [ ]:
def ex3_call_back_fn_validated(back_fn, grad_out, node):
    if grad_out.shape != node.array.shape:
        raise ValueError(
            f'grad_out.shape {tuple(grad_out.shape)} != node.array.shape '
            f'{tuple(node.array.shape)} \u2014 upstream dispatcher bug'
        )
    return back_fn(
        grad_out,
        node.array,
        *node.recipe.args,
        **node.recipe.kwargs,
    )


<details><summary>Solution</summary>

```python
def ex3_call_back_fn_validated(back_fn, grad_out, node):
    if grad_out.shape != node.array.shape:
        raise ValueError(
            f'grad_out.shape {tuple(grad_out.shape)} != node.array.shape '
            f'{tuple(node.array.shape)} \u2014 upstream dispatcher bug'
        )
    return back_fn(
        grad_out,
        node.array,
        *node.recipe.args,
        **node.recipe.kwargs,
    )
```

**`tuple(shape)` is the readable form.** `torch.Size([3, 5])` reprs as `torch.Size([3, 5])` — verbose for an error message. Tupling first yields `(3, 5)` which matches Python literal convention.

**Why this guard is dispatcher-level, not back_fn-level.** The back_fn itself is op-specific (sigmoid_back, sum_back, ...). Shape compatibility between `grad_out` and `out` is INVARIANT across all back_fns — chain rule requires it. Putting the check at the dispatcher avoids duplicating it 30 times in every back_fn.

**Don't validate `args` or `kwargs` shapes here.** Each back_fn knows what shapes its specific args should be (`multiply_back0` needs `y` of broadcastable shape, etc.). Pushing those checks up to the dispatcher would require a per-fn validation table — more code than just letting the back_fn raise its own error.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()